# DD-02 · Android boot: bootloader to launcher

[Open in Colab](https://colab.research.google.com/github/BuiltBySwap/engineering-lab/blob/main/docs/topics/DD-02.ipynb)

**Status card** (edit it; the Progress and Tags pages are built from it):

```yaml
id: DD-02
title: "Android boot: bootloader to launcher"
track: android
date: 2026-10-10
code: ""
status: partial  # not-started | in-progress | partial | done
score: ""
insight: "Each boot stage finds, checks and starts the next (Boot ROM, bootloader, kernel, init, Zygote, system_server, Launcher); init stays alive to restart services, and Zygote makes app launch fast by forking a preloaded process."
tags: [DD-02, android, boot, init, zygote, system-server, verified-boot]
pending:
  - "MQ4 and MQ5 in my own words (skeletons below)"
  - "Three first-principles facts behind the boot order"
  - "English sentences (anomaly, culprit)"
  - "Block A: the DSA timed set"
```

## Goal
Know what exists before my app starts: why Zygote makes launch fast, why system services share one process, and how Verified Boot protects users.

## The chain, in one picture
```text
Boot ROM → Bootloader → Kernel → init → Zygote → system_server → Launcher
 (chip)     (loads the    (Linux)  (PID 1)  (warm      (~100          (an ordinary
            kernel)                          process)   services)       app)
```
**Rule:** each stage finds, checks and starts the **next** one, never the previous one. `init` stays alive to restart services.

**First principles** (three facts force the order): (1) `init` is a program, and only a kernel can create processes, so the kernel comes first; (2) the kernel is a file in storage, so something (the bootloader) must load it; (3) the Launcher is an app, and apps need system services, so it comes last.

## The boot in 10 steps
1. **Power on.** The Boot ROM (code in the chip) runs and checks the bootloader with a key that cannot be changed.
2. **Bootloader.** Picks slot A or B, verifies the boot image (Verified Boot), loads the kernel and the ramdisk, and passes the device tree and command line.
3. **Kernel.** Sets up memory, CPUs, scheduler and drivers (including Android's **Binder** driver), and mounts the ramdisk.
4. **Kernel starts `/init`** as **PID 1**.
5. **init** (two stages): mounts what it needs, sets up SELinux, starts the property service, and reads the `init.rc` files.
6. **init starts the core services** (`servicemanager`, `surfaceflinger`...) and **Zygote** (`class main`).
7. **Zygote** starts the runtime, **preloads** framework classes and resources, and opens its socket.
8. **Zygote forks `system_server`** (the flag `--start-system-server`).
9. **system_server** starts about 100 services, walks through the boot phases, PackageManager scans the apps, and ActivityManager becomes ready.
10. **system_server starts the Launcher**, an ordinary app forked from Zygote that declares the `HOME` category. `sys.boot_completed` becomes `1`; `BOOT_COMPLETED` is sent after the user first unlocks.

## Verified Boot, A/B slots
- **Chain of trust:** the chip's Boot ROM checks the bootloader, the bootloader checks the kernel and boot image, and **dm-verity** checks the system partitions as they are read. Any failed check breaks the chain.
- **Boot states:** green = locked and verified; yellow = verified with a custom key; orange = bootloader unlocked; red = verification failed.
- **A/B slots:** updates install into the unused slot in the background and switch on the next reboot. If the new slot fails to boot, the phone falls back.

## Hands-on 1 · The process tree on the emulator

`adb shell ps -A` (the columns are `USER PID PPID ... NAME`; PPID is the parent). Output from my emulator:

In [ ]:
ps = """
root              1      0 10813836  6236 0                   0 S init
root            295      1 14104708 122816 0                  0 S zygote64
webview_zygote  798    295 14025404 72448 0                   0 S webview_zygote
system          523    295 16751876 231440 0                  0 S system_server
"""

rows = [line.split() for line in ps.strip().splitlines()]
procs = {int(r[1]): (int(r[2]), r[-1]) for r in rows}      # pid -> (parent pid, name)

def show(pid, depth=0):
    print("  " * depth + f"{procs[pid][1]} (PID {pid}, parent {procs[pid][0]})")
    for child, (parent, _) in procs.items():
        if parent == pid:
            show(child, depth + 1)

show(1)
assert procs[295][0] == 1 and procs[523][0] == 295      # Zygote's parent is init; system_server's parent is Zygote

- `init` has PID 1 and parent 0 (the kernel). Zygote's parent is `init`. `system_server` and `webview_zygote` are children of Zygote.
- **PIDs are handed out in creation order**, so Zygote (295) was born before `system_server`. `system_server` gets a new PID on every boot (523, then 578 after a reboot).
- `[kthreadd]` (PID 2) is a kernel thread: the kernel created both PID 1 and PID 2.

## Hands-on 2 · Reading `init.zygote64.rc`
`adb shell cat /system/etc/init/hw/init.zygote64.rc`

| Line | Meaning |
|---|---|
| `service zygote /system/bin/app_process64 -Xzygote /system/bin --zygote --start-system-server` | Run the runtime launcher in Zygote mode and **start `system_server`** |
| `class main` | Started together with the other core services |
| `priority -20` | Highest CPU priority |
| `socket zygote stream 660 root system` | The socket `system_server` uses to ask for new app processes |
| `socket usap_pool_primary ...` | A pool of spare pre-forked app processes (USAP) for faster launches |
| `onrestart restart audioserver` (camera, media, netd, wificond) | If Zygote restarts, restart these too: when Zygote dies, much of the framework restarts with it |
| `writepid /dev/cpuset/foreground/tasks` | Foreground CPU group |

There are four Zygote files (`zygote32`, `zygote64`, `zygote32_64`, `zygote64_32`): a device uses the one that matches its ABIs. `zygote64_32` is a 64-bit primary plus a 32-bit secondary for old apps.

## Hands-on 3 · Timing each stage
`adb reboot`, wait, then `adb logcat -b events -d | grep boot_progress`. Each number is milliseconds since the kernel started. Paste the log here and the cell computes the table:

In [ ]:
import re

log = """
10-12 15:16:31.322   295   295 I boot_progress_start: 9897
10-12 15:16:31.855   295   295 I boot_progress_preload_start: 10430
10-12 15:16:32.322   295   295 I boot_progress_preload_end: 10898
10-12 15:16:32.394   578   578 I boot_progress_system_run: 10969
10-12 15:16:32.688   578   578 I boot_progress_pms_start: 11263
10-12 15:16:32.735   578   578 I boot_progress_pms_system_scan_start: 11310
10-12 15:16:32.832   578   578 I boot_progress_pms_data_scan_start: 11407
10-12 15:16:32.838   578   578 I boot_progress_pms_scan_end: 11413
10-12 15:16:32.861   578   578 I boot_progress_pms_ready: 11436
10-12 15:16:33.589   578   578 I boot_progress_ams_ready: 12164
10-12 15:16:34.778   578   602 I boot_progress_enable_screen: 13353
"""

events = [(m.group(1), int(m.group(2))) for m in re.finditer(r"(boot_progress_\w+): (\d+)", log)]
rows, previous = [], None
for name, ms in events:
    rows.append((name, ms, None if previous is None else ms - previous))
    previous = ms

print(f"{'event':<38}{'ms':>7}{'since previous':>16}")
for name, ms, delta in rows:
    print(f"{name:<38}{ms:>7}{'' if delta is None else delta:>16}")

slowest = max((r for r in rows if r[2] is not None), key=lambda r: r[2])
total = events[-1][1] - events[0][1]
print("\nlongest step:", slowest[0], "took", slowest[2], "ms")
print("from Zygote's start to the screen:", total, "ms")
assert slowest[2] == 1189 and total == 3456

**What my numbers say** (an emulator, so the absolute values are mine only):
- **Longest step: `ams_ready` → `enable_screen`, 1189 ms.** Starting the Launcher, SystemUI and the first screen: real apps launching, forked from Zygote.
- **Next: `pms_ready` → `ams_ready`, 728 ms.** The remaining system services starting.
- **Zygote's preload: 468 ms.** The expensive work done once, so that every later app launch skips it.
- **PackageManager's scan: about 173 ms.** Small here, probably because this emulator image has few apps (my guess, to be tested).
- **Zygote's runtime only started at 9.9 s since the kernel**, so the earlier part (kernel, `init`, early services) was the biggest chunk of all.

## Mobile Exploration (write in my own words)
**MQ4 · How does the Linux kernel initialize Android?**
> The bootloader loads the `___` and the `___` into memory and jumps to the kernel. The kernel sets up `___` and drivers (including Android's `___` driver), mounts the ramdisk, then starts `___` as PID `___`.

**MQ5 · How does init start Android services?**
> `init` reads `___` files made of `___`, `___` and `___`. To start a service it `___` the program, and if the service dies it `___`. The Zygote service has the flag `___`, so after preloading it forks `___`.

## Thinking drill (first principles)
✏️ Three undeniable facts behind the boot order, in my own words.

## English
Words: **anomaly** (something that does not match the normal pattern), **culprit** (the cause of a problem). Grammar: present simple (facts) vs present continuous ("right now").
✏️ Three sentences about the boot, using both words and both tenses.

## Commands used
| Command | What it showed |
|---|---|
| `adb shell ps -A` | the process tree (PID, PPID) |
| `adb shell ls /system/etc/init/hw/` and `cat init.zygote64.rc` | the real Zygote service entry |
| `adb reboot`, then `adb logcat -b events -d \| grep boot_progress` | the time of each boot stage |
| `adb shell getprop sys.boot_completed` | `1` once boot has finished |

## Bug diary
| Error or symptom | Cause | Fix |
|---|---|---|
| I put `init` before the bootloader | I mixed up who starts whom | Each stage starts the **next**; first-principles check: a program needs a kernel first |

## Quiz and mistakes
Right: the first-principles order; Verified Boot (a changed byte fails the check; trust starts in the chip's hardware); `init` needs the kernel; the bootloader loads the kernel; Zygote's preload makes launches fast; the services share one process for fast calls; the Launcher is an ordinary app.

Wrong, and the fix:
- Said Zygote starts `init`: it is the other way round. The kernel starts `init` (PID 1) and `init` starts Zygote.
- Said the bootloader reboots the phone when a service crashes: the bootloader is gone after it loads the kernel; `init` stays alive and restarts the service. (A `critical` service that keeps crashing can make `init` reboot the device.)
- Said the kernel is Zygote's parent, and the bootloader restarts services: the emulator's PPID column settled both.

## In my own words
✏️ The boot chain in three sentences a new Android developer could follow.

## Next
Block A of the plan, the DSA timed set, is still open.